# Module 7 Guided Lab: Aggregation, Reshaping, and Light Validation

**BAN 6003: Data Management and Analytics Integration**

In this lab, we return to the `nycflights13` data. This module is practical: we will use Pandas to summarize data, reshape data, and run light validation checks after those transformations.

The key idea for this week is:

> Aggregation and reshaping can change what each row means.

The original `flights` table has one row per flight. After transformation, a row might mean one carrier, one month, one route, or one carrier-month combination.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Use `groupby()` and `agg()` to summarize flight data.
2. Use `reset_index()` to turn grouped results back into a regular DataFrame.
3. Explain how aggregation changes the unit of analysis.
4. Use `melt()` to reshape wide data into long data.
5. Use `pivot()` or `pivot_table()` to reshape long data into wide data.
6. Run light validation checks after aggregation or reshaping.
7. Write short notes explaining whether the transformed dataset still matches the intended unit of analysis.

We will not use custom functions or complex loops.

## Business Scenario

Imagine you are an analyst helping an airport operations team understand flight delays.

The raw `flights` table has one row per flight. That is useful if we want to inspect individual flights.

But managers usually ask higher-level questions:

- Which carrier has the highest average departure delay?
- Which month has the worst average arrival delay?
- Which origin-destination route has the most flights?
- How do monthly delay patterns compare across carriers?

To answer these questions, we need aggregation and reshaping.

## 0. Setup: Load Packages and Data

The repository requirements include `nycflights13`. In Codespaces they are installed during setup. When working locally, install `requirements.txt` before opening the notebook.

### Reproducibility Note

Package installation is kept outside the analytical workflow. This makes the notebook easier to rerun and avoids hidden environment changes in the middle of an analysis.

In [1]:
import nycflights13
import pandas as pd
from pathlib import Path

/root/.local/lib/python3.12/site-packages/nycflights13/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as _rf


In [2]:
flights = nycflights13.flights.copy()
flights.head()

Out[2]: 
   year  month  day  dep_time  ...  distance  hour  minute             time_hour
0  2013      1    1     517.0  ...      1400     5      15  2013-01-01T10:00:00Z
1  2013      1    1     533.0  ...      1416     5      29  2013-01-01T10:00:00Z
2  2013      1    1     542.0  ...      1089     5      40  2013-01-01T10:00:00Z
3  2013      1    1     544.0  ...      1576     5      45  2013-01-01T10:00:00Z
4  2013      1    1     554.0  ...       762     6       0  2013-01-01T11:00:00Z

[5 rows x 19 columns]


### First reminder: What does one row mean?

Before transforming anything, we need to know the current unit of analysis. In the original `flights` table, each row represents one flight.

In [3]:
flights.shape

Out[3]: (336776, 19)


In [4]:
flights.columns

Out[4]: 
Index(['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay',
       'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight',
       'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute',
       'time_hour'],
      dtype='object')


In [5]:
flights[["year", "month", "day", "carrier", "flight", "origin", "dest", "dep_delay", "arr_delay", "distance", "air_time"]].head()

Out[5]: 
   year  month  day carrier  ...  dep_delay arr_delay distance  air_time
0  2013      1    1      UA  ...        2.0      11.0     1400     227.0
1  2013      1    1      UA  ...        4.0      20.0     1416     227.0
2  2013      1    1      AA  ...        2.0      33.0     1089     160.0
3  2013      1    1      B6  ...       -1.0     -18.0     1576     183.0
4  2013      1    1      DL  ...       -6.0     -25.0      762     116.0

[5 rows x 11 columns]


## 1. Aggregation with `groupby()` and `agg()`

### Business question

Which airline carrier had the highest average departure delay?

To answer this, we need to move from one row per flight to one row per carrier. That is aggregation.

The basic pattern is:

```python
data.groupby("grouping_column").agg({"value_column": "summary_function"})
```

Important arguments:

- The column inside `groupby()` defines the level of the summary.
- The dictionary inside `agg()` says which variable to summarize and how.
- Common summary functions include `"mean"`, `"median"`, `"count"`, `"max"`, `"min"`, and `"std"`.

In [6]:
carrier_delay = flights.groupby("carrier").agg({
    "dep_delay": "mean"
})

carrier_delay.head()

Out[6]: 
         dep_delay
carrier           
9E       16.725769
AA        8.586016
AS        5.804775
B6       13.022522
DL        9.264505


The result is indexed by `carrier`. This is useful, but it does not look like a regular DataFrame yet. In many workflows, we use `reset_index()` after a grouped summary.

In [7]:
carrier_delay = flights.groupby("carrier").agg({
    "dep_delay": "mean"
}).reset_index()

carrier_delay.head()

Out[7]: 
  carrier  dep_delay
0      9E  16.725769
1      AA   8.586016
2      AS   5.804775
3      B6  13.022522
4      DL   9.264505


Now we can sort the summary to see which carriers have the highest average departure delay.

In [8]:
carrier_delay.sort_values("dep_delay", ascending=False).head()

Out[8]: 
   carrier  dep_delay
6       F9  20.215543
5       EV  19.955390
15      YV  18.996330
7       FL  18.726075
14      WN  17.711744


### What changed?

The original data had one row per flight. The aggregated data has one row per carrier.

That means `dep_delay` no longer means "delay for one flight." It now means "average departure delay for that carrier."

This is why aggregation changes the meaning of a row.

### Your Turn 1

Which carrier had the lowest average arrival delay?

Use `groupby()`, `agg()`, `reset_index()`, and `sort_values()`.

In [9]:
# Summarize arrival delay for each carrier, then show the lowest first.
carrier_arrival_delay = flights.groupby("carrier").agg({
    "arr_delay": "mean"
}).reset_index().sort_values("arr_delay")
carrier_arrival_delay

Out[9]: 
   carrier  arr_delay
2       AS  -9.930889
8       HA  -6.915205
1       AA   0.364291
4       DL   1.644341
13      VX   1.764464
12      US   2.129595
11      UA   3.558011
0       9E   7.379669
3       B6   9.457973
14      WN   9.649120
9       MQ  10.774733
10      OO  11.931034
15      YV  15.556985
5       EV  15.796431
7       FL  20.115906
6       F9  21.920705


Write one sentence explaining what one row means in your result.

**Your answer:** Each row represents one carrier and its average arrival delay across flights with a recorded arrival delay.

## 2. Multiple Aggregations

### Business question

For each carrier, what is the average departure delay, the maximum departure delay, and the number of flights?

This is useful because an average alone can hide important operational risk. A carrier may have a moderate average delay but still have some extreme delay cases.

In [10]:
carrier_summary = flights.groupby("carrier").agg(
    avg_dep_delay=("dep_delay", "mean"),
    max_dep_delay=("dep_delay", "max"),
    flight_count=("flight", "count")
).reset_index()

carrier_summary.head()

Out[10]: 
  carrier  avg_dep_delay  max_dep_delay  flight_count
0      9E      16.725769          747.0         18460
1      AA       8.586016         1014.0         32729
2      AS       5.804775          225.0           714
3      B6      13.022522          502.0         54635
4      DL       9.264505          960.0         48110


This syntax is called named aggregation. It lets us choose clear output column names.

The pattern is:

```python
new_column_name=("original_column", "summary_function")
```

This is easier to read than a table with multi-level column names.

In [11]:
carrier_summary.sort_values("avg_dep_delay", ascending=False).head(10)

Out[11]: 
   carrier  avg_dep_delay  max_dep_delay  flight_count
6       F9      20.215543          853.0           685
5       EV      19.955390          548.0         54173
15      YV      18.996330          387.0           601
7       FL      18.726075          602.0          3260
14      WN      17.711744          471.0         12275
0       9E      16.725769          747.0         18460
3       B6      13.022522          502.0         54635
13      VX      12.869421          653.0          5162
10      OO      12.586207          154.0            32
11      UA      12.106073          483.0         58665


### Your Turn 2

Create a monthly delay summary with one row per month.

Your result should include:

- average arrival delay
- median arrival delay
- number of flights

Then sort by average arrival delay from highest to lowest.

In [12]:
monthly_delay_summary = flights.groupby("month").agg(
    avg_arrival_delay=("arr_delay", "mean"),
    median_arrival_delay=("arr_delay", "median"),
    flight_count=("flight", "count")
).reset_index().sort_values("avg_arrival_delay", ascending=False)
monthly_delay_summary

Out[12]: 
    month  avg_arrival_delay  median_arrival_delay  flight_count
6       7          16.711307                  -2.0         29425
5       6          16.481330                  -2.0         28243
11     12          14.870355                   2.0         28135
3       4          11.176063                  -2.0         28330
0       1           6.129972                  -3.0         27004
7       8           6.040652                  -5.0         29327
2       3           5.807577                  -6.0         28834
1       2           5.613019                  -3.0         24951
4       5           3.521509                  -8.0         28796
10     11           0.461347                  -6.0         27268
9      10          -0.167063                  -7.0         28889
8       9          -4.018364                 -12.0         27574


Write one sentence explaining what one row means in your monthly summary.

**Your answer:** Each row represents one month in 2013, with its average and median recorded arrival delay and its total number of flights.

## 3. Aggregating at Two Levels: Carrier and Month

### Business question

Do some carriers experience worse delays in certain months?

Now we want one row per carrier-month combination. This is a different unit of analysis from one row per carrier or one row per month.

In [13]:
carrier_month_delay = flights.groupby(["carrier", "month"]).agg(
    avg_arr_delay=("arr_delay", "mean"),
    flight_count=("flight", "count")
).reset_index()

carrier_month_delay.head()

Out[13]: 
  carrier  month  avg_arr_delay  flight_count
0      9E      1      10.207432          1573
1      9E      2       8.283371          1459
2      9E      3       2.031788          1627
3      9E      4       5.471051          1511
4      9E      5      10.415763          1462


Here, the grouping columns are `carrier` and `month`.

One row now means:

> one carrier in one month

This type of table is often the starting point for dashboards and business reports.

In [14]:
carrier_month_delay.sort_values("avg_arr_delay", ascending=False).head(10)

Out[14]: 
    carrier  month  avg_arr_delay  flight_count
120      OO      1     107.000000             1
121      OO      6      68.500000             2
122      OO      8      64.500000             4
90       FL      7      44.967742           263
178      YV      6      44.589744            49
89       FL      6      41.966805           252
78       F9      7      36.413793            58
75       F9      4      34.315789            57
73       F9      2      31.145833            49
176      YV      4      30.972222            38


### Light validation after aggregation

After aggregation, we should check the basic structure of the result.

For this table, each carrier-month combination should appear only once.

In [15]:
carrier_month_delay.shape

Out[15]: (185, 4)


In [16]:
carrier_month_delay[["carrier", "month"]].duplicated().sum()

Out[16]: np.int64(0)


In [17]:
carrier_month_delay.isna().sum()

Out[17]: 
carrier          0
month            0
avg_arr_delay    0
flight_count     0
dtype: int64


These are light validation checks. We are not doing a full final data audit yet. We are simply checking whether the aggregation result has the structure we intended.

### Your Turn 3

Create a route-level summary with one row per `origin` and `dest`.

Your result should include:

- number of flights
- average arrival delay
- average distance

Then sort to find the busiest routes.

In [18]:
route_summary = flights.groupby(["origin", "dest"]).agg(
    flight_count=("flight", "count"),
    avg_arrival_delay=("arr_delay", "mean"),
    avg_distance=("distance", "mean")
).reset_index().sort_values("flight_count", ascending=False)
route_summary.head(10)

Out[18]: 
    origin dest  flight_count  avg_arrival_delay  avg_distance
117    JFK  LAX         11262          -0.480599        2475.0
156    LGA  ATL         10263          11.322478         762.0
204    LGA  ORD          8857           1.833431         733.0
146    JFK  SFO          8204           2.879640        2586.0
170    LGA  CLT          6168           7.379802         544.0
55     EWR  ORD          6100           8.997255         719.0
92     JFK  BOS          5898           4.913043         187.0
197    LGA  MIA          5781          -0.065766        1096.0
120    JFK  MCO          5464           5.304476         944.0
7      EWR  BOS          5327           4.783686         200.0


Run a light validation check: do any origin-destination pairs appear more than once in your route summary?

In [19]:
route_summary[["origin", "dest"]].duplicated().sum()

Out[19]: np.int64(0)


## 4. Wide vs Long Data

Aggregation summarizes data. Reshaping changes how the same values are arranged.

A wide table uses multiple columns to store different categories or time periods. A long table stores those categories or time periods in a variable column.

Both formats can be useful.

- Wide data is often easier for people to read in a report.
- Long data is often easier for plotting, grouping, and modeling workflows.

## 5. Reshaping Wide to Long with `melt()`

### Business question

Suppose a manager gives us a small report where each month is stored as a separate column. How can we reshape it so each row represents one carrier-month?

We will create a small wide example using carrier-level monthly average arrival delays.

In [20]:
monthly_carrier = flights.groupby(["carrier", "month"]).agg(
    avg_arr_delay=("arr_delay", "mean")
).reset_index()

monthly_carrier.head()

Out[20]: 
  carrier  month  avg_arr_delay
0      9E      1      10.207432
1      9E      2       8.283371
2      9E      3       2.031788
3      9E      4       5.471051
4      9E      5      10.415763


In [21]:
small_summary = monthly_carrier.query("carrier in ['AA', 'DL', 'UA'] and month in [1, 2, 3]")
small_summary

Out[21]: 
    carrier  month  avg_arr_delay
12       AA      1       0.982379
13       AA      2       1.093789
14       AA      3      -2.976651
48       DL      1      -4.404651
49       DL      2      -4.726007
50       DL      3       1.414888
125      UA      1       3.175599
126      UA      2       0.194611
127      UA      3       1.553066


In [22]:
wide_delay = small_summary.pivot(
    index="carrier",
    columns="month",
    values="avg_arr_delay"
).reset_index()

wide_delay

Out[22]: 
month carrier         1         2         3
0          AA  0.982379  1.093789 -2.976651
1          DL -4.404651 -4.726007  1.414888
2          UA  3.175599  0.194611  1.553066


This wide table is easy to read: each row is a carrier, and each month is a separate column.

But if we want to plot month as a variable or group by month again, long format is usually better.

Now we use `melt()`.

In [23]:
long_delay = wide_delay.melt(
    id_vars="carrier",
    var_name="month",
    value_name="avg_arr_delay"
)

long_delay

Out[23]: 
  carrier month  avg_arr_delay
0      AA     1       0.982379
1      DL     1      -4.404651
2      UA     1       3.175599
3      AA     2       1.093789
4      DL     2      -4.726007
5      UA     2       0.194611
6      AA     3      -2.976651
7      DL     3       1.414888
8      UA     3       1.553066


Important arguments in `melt()`:

- `id_vars`: the column or columns that should stay fixed.
- `var_name`: the name of the new column that stores the old column names.
- `value_name`: the name of the new column that stores the old cell values.

After melting, one row means one carrier-month.

### Your Turn 4

Use `melt()` on `wide_delay`.

Create a new long DataFrame called `long_delay_practice`.

Use:

- `carrier` as the ID variable
- `delay_month` as the variable name
- `average_delay` as the value name

In [24]:
long_delay_practice = wide_delay.melt(
    id_vars="carrier",
    var_name="delay_month",
    value_name="average_delay"
)
long_delay_practice

Out[24]: 
  carrier delay_month  average_delay
0      AA           1       0.982379
1      DL           1      -4.404651
2      UA           1       3.175599
3      AA           2       1.093789
4      DL           2      -4.726007
5      UA           2       0.194611
6      AA           3      -2.976651
7      DL           3       1.414888
8      UA           3       1.553066


Run a quick check: how many rows should the long dataset have, based on the number of carriers and month columns?

In [25]:
# Three carriers multiplied by three month columns gives nine rows.
expected_long_rows = len(wide_delay) * (len(wide_delay.columns) - 1)
print("Expected rows:", expected_long_rows)
print("Actual rows:", len(long_delay_practice))
assert len(long_delay_practice) == expected_long_rows
long_delay_practice[["carrier", "delay_month"]].duplicated().sum()

Expected rows: 9
Actual rows: 9
Out[25]: np.int64(0)


## 6. Reshaping Long to Wide with `pivot()` and `pivot_table()`

### Business question

A manager wants a compact comparison table with one row per carrier and one column per month. How do we move from long back to wide?

We can use `pivot()` when each index-column combination has only one value.

In [26]:
wide_again = long_delay.pivot(
    index="carrier",
    columns="month",
    values="avg_arr_delay"
).reset_index()

wide_again

Out[26]: 
month carrier         1         2         3
0          AA  0.982379  1.093789 -2.976651
1          DL -4.404651 -4.726007  1.414888
2          UA  3.175599  0.194611  1.553066


Important arguments in `pivot()`:

- `index`: what should define each row.
- `columns`: which values should become new columns.
- `values`: which numeric values should fill the table.

If there are duplicate combinations, `pivot()` may fail. In that case, `pivot_table()` is often safer because it can aggregate duplicates.

In [27]:
wide_with_pivot_table = small_summary.pivot_table(
    index="carrier",
    columns="month",
    values="avg_arr_delay",
    aggfunc="mean"
).reset_index()

wide_with_pivot_table

Out[27]: 
month carrier         1         2         3
0          AA  0.982379  1.093789 -2.976651
1          DL -4.404651 -4.726007  1.414888
2          UA  3.175599  0.194611  1.553066


### Your Turn 5

Use the `carrier_month_delay` table from earlier.

Create a wide table with:

- one row per carrier
- one column per month
- values equal to average arrival delay

Use `pivot_table()`.

In [28]:
carrier_month_wide = carrier_month_delay.pivot_table(
    index="carrier",
    columns="month",
    values="avg_arr_delay",
    aggfunc="mean"
).reset_index()
carrier_month_wide

Out[28]: 
month carrier           1          2  ...         10         11         12
0          9E   10.207432   8.283371  ...  -1.345566  -0.029243  14.046967
1          AA    0.982379   1.093789  ...  -3.794368  -2.017647   7.719677
2          AS    8.967742 -12.740741  ... -25.919355 -10.807692   7.462963
3          B6    4.717199  12.479341  ...  -2.968327  -0.876667  19.501718
4          DL   -4.404651  -4.726007  ...  -6.295110  -2.760813   6.169904
5          EV   25.160192  20.269003  ...   9.040108   5.087423  26.773066
6          F9   21.830508  31.145833  ...   9.859649  18.216667  22.983607
7          FL    3.317901   4.248227  ...  15.991453  18.539604  29.105769
8          HA   27.483871 -28.821429  ... -14.428571   2.320000  -9.357143
9          MQ    7.883795   7.331045  ...   4.299588   4.847804  14.860664
10         OO  107.000000        NaN  ...        NaN  -1.000000        NaN
11         UA    3.175599   0.194611  ...  -1.525919  -1.302465  14.004565
12         US  

Run a light validation check:

- How many rows does your wide table have?
- Does that match the number of unique carriers?

In [29]:
print("Wide table rows:", len(carrier_month_wide))
print("Unique carriers:", carrier_month_delay["carrier"].nunique())
assert len(carrier_month_wide) == carrier_month_delay["carrier"].nunique()

Wide table rows: 16
Unique carriers: 16


## 7. Light Validation Checklist

After aggregation or reshaping, always ask:

1. What does one row mean now?
2. Did the number of rows change in a way that makes sense?
3. Are the intended keys unique?
4. Did missing values appear or disappear?
5. Do the column names still clearly describe the values?

These checks are small, but they catch many common mistakes.

In [30]:
long_delay[["carrier", "month"]].duplicated().sum()

Out[30]: np.int64(0)


In [31]:
long_delay.isna().sum()

Out[31]: 
carrier          0
month            0
avg_arr_delay    0
dtype: int64


In [32]:
long_delay[["carrier", "month"]].drop_duplicates().shape[0]

Out[32]: 9


## 8. Final Practice: Carrier-Month Dashboard Table

Create a table named `carrier_month_summary` with one row per carrier-month and:

- `avg_departure_delay`
- `avg_arrival_delay`
- `flight_count`
- `avg_distance`

Then create `carrier_month_summary_sorted` with the highest average arrival delays first. Confirm that carrier-month keys are unique and inspect missing values.

In [33]:
carrier_month_summary = flights.groupby(["carrier", "month"]).agg(
    avg_departure_delay=("dep_delay", "mean"),
    avg_arrival_delay=("arr_delay", "mean"),
    flight_count=("flight", "count"),
    avg_distance=("distance", "mean")
).reset_index()
carrier_month_summary.head()

Out[33]: 
  carrier  month  ...  flight_count  avg_distance
0      9E      1  ...          1573    476.354100
1      9E      2  ...          1459    467.893077
2      9E      3  ...          1627    476.488629
3      9E      4  ...          1511    492.468564
4      9E      5  ...          1462    505.490424

[5 rows x 6 columns]


In [34]:
carrier_month_summary_sorted = carrier_month_summary.sort_values(
    "avg_arrival_delay", ascending=False
)
carrier_month_summary_sorted.head(10)

Out[34]: 
    carrier  month  ...  flight_count  avg_distance
120      OO      1  ...             1    733.000000
121      OO      6  ...             2    488.000000
122      OO      8  ...             4    419.000000
90       FL      7  ...           263    644.034221
178      YV      6  ...            49    389.714286
89       FL      6  ...           252    646.126984
78       F9      7  ...            58   1620.000000
75       F9      4  ...            57   1620.000000
73       F9      2  ...            49   1620.000000
176      YV      4  ...            38    403.078947

[10 rows x 6 columns]


In [35]:
carrier_month_summary[["carrier", "month"]].duplicated().sum()

Out[35]: np.int64(0)


In [36]:
carrier_month_summary.isna().sum()

Out[36]: 
carrier                0
month                  0
avg_departure_delay    0
avg_arrival_delay      0
flight_count           0
avg_distance           0
dtype: int64


In [37]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

if carrier_month_summary is None:
    print("Complete Final Practice Step 1 before exporting.")
else:
    module7_output_path = output_dir / "carrier_month_summary_module7.csv"
    carrier_month_summary.to_csv(module7_output_path, index=False)
    print(f"Saved {len(carrier_month_summary)} rows to: {module7_output_path}")


Saved 185 rows to: ../data/processed/carrier_month_summary_module7.csv


### Final Reflection

**Your reflection:**  
Each row in `carrier_month_summary` represents one carrier operating in one month of 2013. The `groupby()` aggregation changed the row meaning the most because it replaced individual flight records with group averages and counts. The result has 185 carrier-month rows, zero duplicate keys, and no missing summary values; its flight counts also add back to all 336,776 original flights. There are fewer than 192 combinations because not every carrier operated in every month. An operations team could use this table to flag carrier-month combinations with high average arrival delays and check their flight volumes before deciding where to investigate.

## 9. Offline Assignment - Route Performance Summary

An airport operations manager wants to review established routes rather than individual flights.

1. Create `route_performance` with one row per `origin`-`dest` pair.
2. Include flight count, average arrival delay, and average distance using named aggregation.
3. Keep routes with at least 500 flights and sort from highest to lowest average arrival delay.
4. Verify that route keys are unique and inspect missing values.
5. Save `data/processed/route_performance_module7.csv`.

In [38]:
# Summarize first, then restrict the review to established routes.
route_performance = flights.groupby(["origin", "dest"], as_index=False).agg(
    flight_count=("flight", "count"),
    avg_arrival_delay=("arr_delay", "mean"),
    avg_distance=("distance", "mean")
)
route_performance = route_performance.query("flight_count >= 500").sort_values(
    "avg_arrival_delay", ascending=False
).reset_index(drop=True)

print("Routes retained:", len(route_performance))
print("Duplicate route keys:", route_performance[["origin", "dest"]].duplicated().sum())
print("Missing values:")
print(route_performance.isna().sum())
display(route_performance.head(10))

route_output_path = output_dir / "route_performance_module7.csv"
route_performance.to_csv(route_output_path, index=False)
print(f"Saved {len(route_performance)} rows to: {route_output_path}")

Routes retained: 138
Duplicate route keys: 0
Missing values:
origin               0
dest                 0
flight_count         0
avg_arrival_delay    0
avg_distance         0
dtype: int64
Saved 138 rows to: ../data/processed/route_performance_module7.csv


,origin,dest,flight_count,avg_arrival_delay,avg_distance
0,EWR,RIC,1667,25.820433,277.0
1,EWR,MKE,1094,24.003781,725.0
2,EWR,PWM,770,23.005479,284.0
3,JFK,CMH,742,22.920567,483.0
4,EWR,DCA,1719,21.313976,199.0
5,EWR,ROC,513,20.663934,246.0
6,EWR,MCI,1356,20.239812,1092.0
7,EWR,MEM,926,19.907692,946.0
8,LGA,CAK,864,19.698337,397.0
9,EWR,GRR,719,19.370640,605.0


### Route Summary Interpretation

**Your response:**  
Each row in `route_performance` represents one origin-destination route, with its flight count, average recorded arrival delay, and average distance. The 500-flight filter keeps 138 of the 224 routes, so this review focuses on routes with more observations and excludes lower-volume routes. The retained table has zero duplicate route keys and no missing values in its summary columns. EWR to RIC has the highest average arrival delay among the retained routes, about 25.82 minutes across 1,667 flights. A manager could prioritize that route for a closer operational review, but this summary alone does not tell us what caused its delays.

## 10. Save Your Work

Before submitting:

1. Complete all Your Turn and final-practice cells.
2. Complete the offline route summary and both reflections.
3. Restart the kernel and run all cells from top to bottom.
4. Confirm both Module 7 CSV files exist in `data/processed`.
5. Save the notebook, then commit and push your work.

Suggested commit message:

```bash
git add .
git commit -m "Complete Module 7 aggregation and reshaping lab"
git push
```